# House Prices : Linear Regression RMSE 0.109

Two houses in Ames, Iowa sit on the same street. One sells for \$130,000, the other for \$340,000. A buyer sees bricks and windows; a model sees 79 columns. This notebook builds linear regression from one feature to all of them and lets the **formula** decide each step: which columns to log, which to one-hot, which to drop. Only then does a tree model get a turn as a benchmark.

**Objectives**
1. How are prices distributed, and why does the competition score log prices?
2. How much does one extra square foot add, and what does a regression line actually learn?
3. Which linear regression assumptions break on this data, and what fixes them?
4. How far do all 79 columns take plain linear regression, and why does it need Ridge / Lasso?
5. Does a gradient-boosted tree model beat the best linear model?
6. Does trimming outliers help, and which of 6 treatments works best?

**Data** → [Kaggle competition](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques) · 1,460 sold houses with prices · 1,459 houses to price · 79 features

# [1/12] Setup

Set every knob once in the top cell → later cells only read them, and nothing is written by default

In [ ]:
"""Optional · Edit the settings, then run all cells. Explanations: README.md

[1/13] DATA_DIR: competition data folder
    Default = '/kaggle/input/house-prices-advanced-regression-techniques'
[2/13] WORK_DIR: folder for the submission file
    Default = '/kaggle/working'
[3/13] N_FOLDS: cross-validation folds
    Default = 5
[4/13] SEED: random seed
    Default = 42
[5/13] DROP_PARTIAL_GIANTS: drop training houses above 4,000 sq ft sold as partial sales
    Default = True
[6/13] TRIM_OUTLIERS: outlier treatment of TRIM_COLUMNS inside every model; None keeps the values
    Default = None
    For example:
        TRIM_OUTLIERS = 'zscore'      (or 'iqr', 'winsorize')
[7/13] TRIM_COLUMNS: {column: alpha} checked by the trimmer; alpha = share a z-score flags on a bell curve
    Default = LotArea, TotalBsmtSF, 1stFlrSF, GrLivArea 0.05 · LotFrontage 0.01
[8/13] IQR_K / WINSOR_LIMITS: IQR fence width / percentiles where winsorize caps values
    Default = 1.5 / (0.01, 0.99)
[9/13] SKEW_LIMIT: |skewness| above which a numeric feature gets log1p
    Default = 0.75
[10/13] MIN_CATEGORY_COUNT: categories rarer than this are merged into one "infrequent" column
    Default = 10
[11/13] RIDGE_ALPHAS / LASSO_ALPHAS / HUBER_ALPHAS: (start, stop, count) of the log10 grid of penalty strengths
    Default = (-1, 3, 25) / (-4.5, -1.5, 25) / (1, 2.5, 7)
[12/13] LGBM_PARAMS: LightGBM benchmark settings
    Default = 2,000 trees of 15 leaves, learning rate 0.01
[13/13] SAVE_SUBMISSION: predict all test houses and write submission.csv
    Default = False
    For example:
        SAVE_SUBMISSION = True
"""
DATA_DIR            = '/kaggle/input/house-prices-advanced-regression-techniques'
WORK_DIR            = '/kaggle/working'
N_FOLDS             = 5
SEED                = 42
DROP_PARTIAL_GIANTS = True
TRIM_OUTLIERS       = None
TRIM_COLUMNS        = {
    'LotArea':     0.05,
    'LotFrontage': 0.01,
    'TotalBsmtSF': 0.05,
    '1stFlrSF':    0.05,
    'GrLivArea':   0.05,
}
IQR_K               = 1.5
WINSOR_LIMITS       = (0.01, 0.99)
SKEW_LIMIT          = 0.75
MIN_CATEGORY_COUNT  = 10
RIDGE_ALPHAS        = (-1, 3, 25)
LASSO_ALPHAS        = (-4.5, -1.5, 25)
HUBER_ALPHAS        = (1, 2.5, 7)
LGBM_PARAMS         = {
    'n_estimators':      2000,
    'learning_rate':     0.01,
    'num_leaves':        15,
    'min_child_samples': 10,
    'subsample':         0.7,
    'subsample_freq':    1,
    'colsample_bytree':  0.5,
    'reg_lambda':        1.0,
    'verbose':           -1,
}
SAVE_SUBMISSION     = False

SUBMISSION_FILE     = f'{WORK_DIR}/submission.csv'
SUBMISSION_PREVIEW  = 10
TARGET              = 'SalePrice'
PAD                 = 30
PALETTE             = {'main': 'black', 'flag': 'red', 'soft': 'grey'}
QUALITY_SCALE       = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1}
QUALITY_COLS        = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 'KitchenQual',
                       'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']
ABSENT_MEANS_NONE   = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType', 'FireplaceQu',
                       'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
                       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from scipy.stats import skew, probplot, norm

from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools import add_constant

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, HuberRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from lightgbm import LGBMRegressor

import warnings

sns.set_theme(style='darkgrid', palette='gray')
warnings.filterwarnings('ignore', message='X does not have valid feature names')
warnings.filterwarnings('ignore', message='Precision loss occurred in moment calculation')

In [ ]:
def show_block(title, items):
    print(f'{title}:')
    for key, value in items.items():
        print(f'    {key:<{PAD}}{value}')


def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))


def qq_plot(values, ax, title):
    (theoretical, ordered), (slope, intercept, _) = probplot(values, dist='norm')
    ax.scatter(theoretical, ordered, s=8, color=PALETTE['main'])
    ax.plot(theoretical, slope * theoretical + intercept, color=PALETTE['flag'])
    ax.set(title=title, xlabel='Normal quantiles', ylabel='Ordered values')

# [2/12] Data

Load the CSVs and read what the gaps mean → most "missing" values say the house has no pool, alley, fence, garage or basement

In [ ]:
from pathlib import Path


def find_competition_dir(default, required_column, root='/kaggle/input'):
    """Folder with train.csv + test.csv whose train.csv has `required_column`.
    Tries `default` first, then searches everything under `root` (Kaggle mount paths differ between setups)."""
    import csv
    candidates = [Path(default)]
    if Path(root).exists():
        candidates += sorted(p.parent for p in Path(root).rglob('train.csv'))
    for folder in candidates:
        if (folder / 'train.csv').is_file() and (folder / 'test.csv').is_file():
            with open(folder / 'train.csv', newline='') as fh:
                if required_column in next(csv.reader(fh)):
                    return str(folder)
    mounted = sorted(str(p) for p in Path(root).glob('*/*'))[:20] if Path(root).exists() else []
    raise FileNotFoundError(
        f"(!) No train.csv with a '{required_column}' column under {default} or {root}.\n"
        f"    Add Input -> Competitions -> {Path(default).name} (join the competition and accept its rules first)\n"
        f"    Mounted now: {mounted or 'nothing'}")


DATA_DIR = find_competition_dir(DATA_DIR, required_column='SalePrice')

train_raw = pd.read_csv(f'{DATA_DIR}/train.csv')
test_raw  = pd.read_csv(f'{DATA_DIR}/test.csv')
show_block('Outputs', {'train_raw': train_raw.shape, 'test_raw': test_raw.shape})

In [ ]:
display(train_raw.head())

**Question:** 19 training columns have gaps, some 99% empty. Is that broken data we should drop?

**Answer:** No. Only 3 of the 19 are real unknowns: `LotFrontage` (17.7%), `MasVnrArea` (8 houses) and `Electrical` (1 house). The other 16 are the data dictionary's way of writing **"none"**:
- `PoolQC` is empty for 1,453 houses because 1,453 houses have no pool
- all 5 garage columns are empty for the same 81 garage-less houses, all 5 basement columns for the same ~37 basement-less ones
- `MasVnrType`: pandas reads the text `"None"` as missing, so its 872 gaps are houses without masonry veneer

→ Fill them with `'None'` (or 0) instead of dropping the column: dropping `PoolQC` would throw away "this house has a pool". The 3 real gaps are imputed **inside** cross-validation (median), so validation houses never leak into the fill value.

In [ ]:
missing = pd.DataFrame({'train missing': train_raw.isna().sum(), 'test missing': test_raw.isna().sum()})
missing = missing[(missing > 0).any(axis=1)].fillna(0).astype(int)
missing['train %'] = (missing['train missing'] / len(train_raw) * 100).round(1)
missing['meaning'] = np.where(missing.index.isin(ABSENT_MEANS_NONE) | missing.index.str.startswith(('Garage', 'Bsmt')),
                              'house has none', 'unknown')
display(missing.sort_values('train missing', ascending=False))

# [3/12] Target

Look at the price distribution and the competition metric → model log price: it is symmetric and it is what Kaggle scores

**Question:** What does a typical price look like, and why work with its logarithm?

$$\tilde y = \log(1 + y)$$
- $y$ → sale price in dollars
- $\tilde y$ → log price: equal **ratios** become equal **steps** (\$100k → \$200k is the same step as \$200k → \$400k)
- Analogy → a salary raise: +\$5,000 is huge on \$20,000 and barely noticed on \$200,000; people think in percent, and so does the log

$$\text{RMSE}_{\log} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}\big(\log(1+\hat y_i) - \log(1+y_i)\big)^2}$$
- $\hat y_i$ → predicted price of house $i$, $y_i$ → its real price, $n$ → number of houses
- the competition metric: errors are measured in **percent**, so a \$20k miss on a \$100k house costs as much as a \$100k miss on a \$500k house
- Analogy → grading a weather forecast by "how many times off" rather than "how many degrees off"

**Answer:** A typical house sells for **\$163,000** (median); the mean is higher, \$180,921, because a long tail of expensive houses (up to \$755,000) pulls it up. That tail gives skewness **1.88**, and in squared errors a few mansions would outweigh hundreds of normal houses. After $\log(1+y)$ the skewness drops to **0.12** and the Q-Q plot hugs the line → every model below predicts log price, and RMSE on log price is exactly the competition score.

In [ ]:
target_stats = pd.DataFrame({'SalePrice':        train_raw[TARGET].describe(),
                             'log1p(SalePrice)': np.log1p(train_raw[TARGET]).describe()})
target_stats.loc['skewness'] = [skew(train_raw[TARGET]), skew(np.log1p(train_raw[TARGET]))]
display(target_stats.round(3))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for col, (name, values) in enumerate([('SalePrice ($)', train_raw[TARGET]), ('log1p(SalePrice)', np.log1p(train_raw[TARGET]))]):
    sns.histplot(values, bins=50, kde=True, color=PALETTE['main'], ax=axes[0, col])
    axes[0, col].set_title(f'{name} · skewness {skew(values):.2f}')
    qq_plot(values, axes[1, col], f'Q-Q plot · {name}')
plt.tight_layout()
plt.show()

# [4/12] Simple linear regression

Fit one feature, living area, by hand and by gradient descent → a price per square foot, and 2 houses that bend the line

**Question:** How much does one extra square foot of living area add to the price?

$$\hat y = \beta_0 + \beta_1 x$$
- $x$ → `GrLivArea`, living area above ground in sq ft
- $\hat y$ → predicted sale price
- $\beta_1$ → **slope**: dollars added per extra square foot. Analogy → a taxi meter: every extra kilometre adds the same fare
- $\beta_0$ → **intercept**: the predicted price at $x = 0$. Analogy → the taxi's flag-fall fee you pay before moving; for a house of 0 sq ft it has no real meaning, it only anchors the line

$$\beta_1 = \frac{\sum_i (x_i - \bar x)(y_i - \bar y)}{\sum_i (x_i - \bar x)^2} \qquad \beta_0 = \bar y - \beta_1 \bar x$$
- $\bar x, \bar y$ → average area and average price
- numerator → do area and price rise **together**? (positive when bigger-than-average houses are pricier-than-average)
- denominator → how spread out the areas are
- Analogy → a class's study hours vs exam scores: the slope is "points gained per extra hour", and the line always passes through the class average

**Answer:** **\$107 per square foot**, on top of an intercept of \$18,569; the formula and scikit-learn agree to the cent. Living area alone explains 54% of the price variation (R² = 0.54 in [5/12]). The 4 red houses above 4,000 sq ft don't share one story, though: 2 sold for \$745k-755k, as the line expects, and 2 sold for only \$160k-185k.

In [ ]:
def slr(x, y):
    x, y   = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    beta_1 = np.sum((x - x.mean()) * (y - y.mean())) / np.sum((x - x.mean()) ** 2)
    return y.mean() - beta_1 * x.mean(), beta_1


beta_0, beta_1 = slr(train_raw['GrLivArea'], train_raw[TARGET])
sk_model       = LinearRegression().fit(train_raw[['GrLivArea']], train_raw[TARGET])
display(pd.DataFrame({'by formula':   [beta_0, beta_1],
                      'scikit-learn': [sk_model.intercept_, sk_model.coef_[0]]},
                     index=['β0 intercept ($)', 'β1 slope ($ per sq ft)']).round(2))

In [ ]:
giants         = train_raw['GrLivArea'] > 4000
partial_giants = giants & (train_raw['SaleCondition'] == 'Partial')
grid           = np.linspace(train_raw['GrLivArea'].min(), train_raw['GrLivArea'].max(), 100)

fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter(train_raw.loc[~giants, 'GrLivArea'], train_raw.loc[~giants, TARGET], s=8, alpha=0.5, color=PALETTE['soft'], label='houses')
ax.scatter(train_raw.loc[giants, 'GrLivArea'], train_raw.loc[giants, TARGET], s=60, color=PALETTE['flag'], label='> 4,000 sq ft')
ax.plot(grid, beta_0 + beta_1 * grid, color=PALETTE['main'], label=f'ŷ = {beta_0:,.0f} + {beta_1:,.1f} · x')
ax.set(xlabel='GrLivArea (sq ft)', ylabel='SalePrice ($)', title='Simple linear regression: living area → price')
ax.legend()
plt.show()

display(train_raw.loc[giants, ['Id', 'GrLivArea', 'OverallQual', 'SaleCondition', TARGET]])

**Question:** Should those 2 giant houses at bargain prices stay in training?

**Answer:** No. Both are **partial sales** (`SaleCondition = Partial`: sold before construction was finished), so their price does not describe the finished house. The dataset's author, Dean De Cock, recommends removing the houses above 4,000 sq ft for this reason. Removing only these 2 moves the slope from \$107 to **\$115 per sq ft** and the intercept from \$18,569 to \$7,169: 2 houses out of 1,460 bent the whole line, like one faulty scale at a market changing what every customer pays. The 2 genuine giants (\$745k and \$755k) stay.

In [ ]:
train = train_raw.loc[~partial_giants].reset_index(drop=True) if DROP_PARTIAL_GIANTS else train_raw.copy()
beta_0_clean, beta_1_clean = slr(train['GrLivArea'], train[TARGET])
display(pd.DataFrame({'all houses':         [beta_0, beta_1, len(train_raw)],
                      'without 2 partials': [beta_0_clean, beta_1_clean, len(train)]},
                     index=['β0 intercept ($)', 'β1 slope ($ per sq ft)', 'houses']).round(2))

**Question:** scikit-learn solved the line in one formula. How do models that have no such formula find it?

$$J(w, b) = \frac{1}{2n}\sum_{i=1}^{n}\big(w x_i + b - y_i\big)^2$$
- $w, b$ → slope and intercept being searched for
- $J$ → **cost**: the average squared miss; a bowl-shaped surface over all $(w, b)$ pairs

$$w \leftarrow w - \eta\,\frac{\partial J}{\partial w} \qquad b \leftarrow b - \eta\,\frac{\partial J}{\partial b}$$
- $\frac{\partial J}{\partial w}, \frac{\partial J}{\partial b}$ → **gradient**: which direction makes the miss grow fastest
- $\eta$ → **learning rate**: step size
- Analogy → walking down a hill in thick fog: feel which way the ground slopes under your feet, take a step the other way, repeat. Too-big steps overshoot the valley, too-small ones take all day

Both $x$ and $\log$ price are standardised first (mean 0, std 1) so the bowl is round and one learning rate suits both directions.

**Answer:** Same answer, different road. After 100 steps gradient descent lands on slope **0.7252** and intercept **0.0000**, identical to the closed formula to 4 decimals, with the same cost J = 0.2369. The path runs straight down the bowl and the cost stops moving after about 30 steps. On standardised data the slope is simply the correlation between area and log price: 0.73.

In [ ]:
def gradient_descent(x, y, lr=0.1, steps=100):
    w, b, path = 0.0, 0.0, []
    for _ in range(steps):
        error = w * x + b - y
        path.append((w, b, np.mean(error ** 2) / 2))
        w    -= lr * np.mean(error * x)
        b    -= lr * np.mean(error)
    return w, b, np.array(path)


x_std          = ((train['GrLivArea'] - train['GrLivArea'].mean()) / train['GrLivArea'].std()).to_numpy()
y_std          = ((np.log1p(train[TARGET]) - np.log1p(train[TARGET]).mean()) / np.log1p(train[TARGET]).std()).to_numpy()
w_gd, b_gd, gd_path = gradient_descent(x_std, y_std)
b_ls, w_ls     = slr(x_std, y_std)
display(pd.DataFrame({'gradient descent (100 steps)': [w_gd, b_gd, gd_path[-1, 2]],
                      'closed formula':               [w_ls, b_ls, np.mean((w_ls * x_std + b_ls - y_std) ** 2) / 2]},
                     index=['w slope', 'b intercept', 'cost J']).round(4))

In [ ]:
W, B = np.meshgrid(np.linspace(-0.25, 1.25, 80), np.linspace(-0.75, 0.75, 80))
J    = ((W[..., None] * x_std + B[..., None] - y_std) ** 2).mean(axis=-1) / 2

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
contour = axes[0].contour(W, B, J, levels=15, colors=PALETTE['soft'], linewidths=1)
axes[0].clabel(contour, fontsize=8)
axes[0].plot(gd_path[:, 0], gd_path[:, 1], marker='o', markersize=3, color=PALETTE['flag'], label='gradient descent path')
axes[0].scatter([w_ls], [b_ls], marker='*', s=250, color=PALETTE['main'], zorder=3, label='closed-formula answer')
axes[0].set(xlabel='w (slope)', ylabel='b (intercept)', title='Cost surface J(w, b): walking downhill')
axes[0].legend()
axes[1].plot(gd_path[:, 2], color=PALETTE['main'])
axes[1].set(xlabel='step', ylabel='cost J', title='Cost per step')
plt.tight_layout()
plt.show()

# [5/12] Assumptions

Test the textbook assumptions on the one-feature model → log price fixes the funnel-shaped errors; overlapping features call for regularisation

**Question:** Linear regression promises honest coefficients only if 5 assumptions hold. Which ones break here?

1. **Linearity** → price rises in a straight line with each feature
2. **Independence** → one house's error tells nothing about another's
3. **Homoscedasticity** → errors are equally large for cheap and expensive houses
4. **Normality** → errors follow a bell curve, needed for trustworthy p-values and intervals
5. **No multicollinearity** → features don't repeat each other

Residual of house $i$ → $e_i = y_i - \hat y_i$, what the line got wrong. Analogy → a tailor's fitting: the residual is how far each sleeve ends from the wrist. Good tailoring leaves small, random gaps, not sleeves that get worse the longer the arm.

**Breusch-Pagan test** → fits a small regression of $e_i^2$ on the features: if the features can predict how big the error is, the errors are not equally sized. p-value below 0.05 → heteroscedasticity.

**Answer (1-4):** 
- **Linearity** → holds well enough: area and price rise together in one straight band, and switching to log price costs almost no fit (R² 0.54 → 0.53)
- **Independence** → each row is a separate sale, not a time series, so there is no order through which errors could leak into each other
- **Homoscedasticity** → broken on raw prices: the residuals fan out like a funnel as prices rise (Breusch-Pagan LM = 249). On log price the funnel becomes an even band, LM = 23, about 11× smaller (still significant with a single feature)
- **Normality** → raw residuals skew right (0.80): the line under-prices expensive houses. Log residuals are close to the bell curve except a left tail (−0.55) of cheap houses priced too high; they return in [11/12]

→ From here on every model predicts log price.

In [ ]:
fits = {'SalePrice':        train[TARGET].to_numpy(dtype=float),
        'log1p(SalePrice)': np.log1p(train[TARGET]).to_numpy()}
X_one = train[['GrLivArea']].to_numpy(dtype=float)
rows  = {}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for col, (name, target) in enumerate(fits.items()):
    model  = LinearRegression().fit(X_one, target)
    fitted = model.predict(X_one)
    resid  = target - fitted
    rows[name] = {'R²':                model.score(X_one, target),
                  'residual skewness': skew(resid),
                  'Breusch-Pagan LM':  het_breuschpagan(resid, add_constant(X_one))[0],
                  'Breusch-Pagan p':   het_breuschpagan(resid, add_constant(X_one))[1]}
    axes[0, col].scatter(fitted, resid, s=8, alpha=0.5, color=PALETTE['main'])
    axes[0, col].axhline(0, color=PALETTE['flag'])
    axes[0, col].set(title=f'Residuals vs fitted · {name}', xlabel='fitted', ylabel='residual')
    qq_plot(resid, axes[1, col], f'Q-Q plot of residuals · {name}')
plt.tight_layout()
plt.show()

display(pd.DataFrame(rows).T.round(4))

**Question:** Do the strongest features repeat each other (assumption 5)?

$$\text{VIF}_j = \frac{1}{1 - R_j^2}$$
- $R_j^2$ → how well feature $j$ is predicted by **all the other** features
- $\text{VIF}_j$ → **variance inflation factor**: how many times wider the uncertainty of $\beta_j$ becomes because of that overlap. 1 = independent, above 5 = worrying, above 10 = severe
- Analogy → two witnesses who rode in the same car: they tell the same story, so the court can't tell whose testimony matters; the regression can't split the credit between `GarageCars` and `GarageArea` either

**Answer:** Moderate overlap, nothing severe: the highest VIF is **5.4** (`GarageCars`), then `GrLivArea` 5.0 and `GarageArea` 5.0. Garage size is counted twice, once in cars and once in square feet (correlation 0.88), and living area overlaps with the room count (0.83). No single VIF passes 10, but with all 79 columns and almost 200 one-hot categories the small overlaps add up → instead of dropping columns by hand, [8/12] lets Ridge and Lasso share or remove the credit.

In [ ]:
vif_cols = ['OverallQual', 'GrLivArea', 'TotRmsAbvGrd', 'GarageCars', 'GarageArea',
            'TotalBsmtSF', '1stFlrSF', 'FullBath', 'YearBuilt', 'YearRemodAdd']
vif_X    = add_constant(train[vif_cols].fillna(0).astype(float))
vif      = pd.DataFrame({'VIF': [variance_inflation_factor(vif_X.values, i) for i in range(1, vif_X.shape[1])]}, index=vif_cols)
display(vif.sort_values('VIF', ascending=False).round(2))

# [6/12] Features

Let the formula decide each transform → ordinal scales become numbers, nominal categories become one-hot columns, skewed areas get log1p

**Question:** A regression multiplies every column by one $\beta$. What does that mean for text columns?

**1. Ordinal scales** (`Ex > Gd > TA > Fa > Po`) → numbers 5 … 1, absent = 0

$$\hat y = \dots + \beta_{\text{qual}} \cdot \text{quality}$$
- one $\beta$ means every step up the scale adds the **same** amount
- Analogy → hotel stars: each extra star costs roughly the same premium, so one price per star is a fair summary

**2. Nominal categories** (`Neighborhood`, `MSZoning`, …) → one-hot: one 0/1 column, and one $\beta$, per category

$$\hat y = \dots + \beta_{\text{NridgHt}} \cdot [\text{Neighborhood} = \text{NridgHt}] + \beta_{\text{OldTown}} \cdot [\text{Neighborhood} = \text{OldTown}] + \dots$$
- label encoding (`MeadowV = 1, NridgHt = 2, …`) would force equal, ordered steps between neighbourhoods that have no order
- Analogy → train stations: a line numbered 1-2-3 doesn't mean station 3 is "three times" station 1; each stop needs its own fare
- `MSSubClass` is a building-type **code** stored as numbers → treated as nominal

**3. Skewed numeric features** → $\log(1 + x)$ when $|\text{skewness}| >$ `SKEW_LIMIT`, then standardised

- a few huge lots or porches would otherwise steer the line, exactly as the 2 giant houses did
- the skewness is measured inside each training fold, never on validation rows

**Answer:** The data agrees with the formula:
- `ExterQual` → median price \$82k → \$139k → \$220k → \$369k from Fair to Excellent: each step multiplies the price by about 1.6-1.7, i.e. **equal steps on the log scale**, exactly what one $\beta$ per step on log price assumes
- `Neighborhood` → 25 neighbourhoods jump unevenly from \$88,000 (MeadowV) to \$315,000 (NridgHt), with no order in their names → one $\beta$ each

Result: 55 numeric columns (31 of them log-transformed because |skewness| > 0.75) and 34 categorical columns that become 194 one-hot columns → a model matrix of **1,458 houses × 249 columns**.

In [ ]:
qual_price  = (train.assign(level=train['ExterQual'].map(QUALITY_SCALE))
               .groupby('level')[TARGET].agg(['median', 'size']))
neigh_price = train.groupby('Neighborhood')[TARGET].agg(['median', 'size']).sort_values('median')

fig, axes = plt.subplots(1, 2, figsize=(17, 5), gridspec_kw={'width_ratios': [1, 3]})
axes[0].bar(qual_price.index.astype(str), qual_price['median'], color=PALETTE['main'])
axes[0].set(title='ExterQual: ordered steps', xlabel='quality level (2 = Fa … 5 = Ex)', ylabel='median SalePrice ($)')
axes[1].bar(neigh_price.index, neigh_price['median'], color=PALETTE['soft'])
axes[1].tick_params(axis='x', rotation=60)
axes[1].set(title='Neighborhood: no natural order, uneven gaps', ylabel='median SalePrice ($)')
plt.tight_layout()
plt.show()

**Question:** Which new features can we build from the existing columns, and do they help?

| Feature | Built from | Why |
|---|---|---|
| `TotalSF` | basement + 1st + 2nd floor | buyers pay for total space, not per floor |
| `TotalBath` | full + ½ half baths, above ground and basement | one "bathroom count" instead of 4 overlapping ones |
| `TotalPorchSF` | open, enclosed, 3-season, screen porch, wood deck | 5 sparse columns → 1 dense one |
| `HouseAge`, `RemodAge` | `YrSold − YearBuilt`, `YrSold − YearRemodAdd` | age **at sale**, not years before an arbitrary date |
| `Has…` flags | pool, 2nd floor, garage, basement, fireplace | 0 sq ft means "none", not "very small" |

**Answer:** Yes. `TotalSF` becomes the strongest single feature in the data: correlation **0.83** with log price, against 0.73 for `GrLivArea` and 0.65 for `TotalBsmtSF`. `TotalBath` (0.68) beats `FullBath` (0.60), and `TotalPorchSF` (0.40) beats the best single porch column (0.33). The ages mirror their year columns (−0.59 vs 0.59): the same information, read as years of age at the sale. The `Has…` flags are weaker on their own (0.08-0.51); they mainly let the linear model tell "none" apart from "small".

In [ ]:
def prepare(df):
    out = df.drop(columns=['Id', TARGET], errors='ignore').copy()
    for col in ABSENT_MEANS_NONE:
        out[col] = out[col].fillna('None')
    for col in QUALITY_COLS:
        out[col] = out[col].map(QUALITY_SCALE).fillna(0)
    out['MSSubClass']   = out['MSSubClass'].astype(str)
    out['GarageYrBlt']  = out['GarageYrBlt'].fillna(out['YearBuilt'])
    out['TotalSF']      = out['TotalBsmtSF'].fillna(0) + out['1stFlrSF'] + out['2ndFlrSF']
    out['TotalBath']    = (out['FullBath'] + 0.5 * out['HalfBath']
                           + out['BsmtFullBath'].fillna(0) + 0.5 * out['BsmtHalfBath'].fillna(0))
    out['TotalPorchSF'] = out[['OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'WoodDeckSF']].sum(axis=1)
    out['HouseAge']     = out['YrSold'] - out['YearBuilt']
    out['RemodAge']     = out['YrSold'] - out['YearRemodAdd']
    out['HasPool']      = (out['PoolArea'] > 0).astype(int)
    out['Has2ndFlr']    = (out['2ndFlrSF'] > 0).astype(int)
    out['HasGarage']    = (out['GarageArea'].fillna(0) > 0).astype(int)
    out['HasBsmt']      = (out['TotalBsmtSF'].fillna(0) > 0).astype(int)
    out['HasFireplace'] = (out['Fireplaces'] > 0).astype(int)
    return out


X_all    = prepare(train)
X_test   = prepare(test_raw)
y_log    = np.log1p(train[TARGET]).to_numpy()
num_cols = [col for col in X_all.columns if pd.api.types.is_numeric_dtype(X_all[col])]
cat_cols = [col for col in X_all.columns if col not in num_cols]
show_block('Outputs', {'X_all':                X_all.shape,
                       'X_test':               X_test.shape,
                       'numeric columns':      len(num_cols),
                       'categorical columns':  len(cat_cols)})

In [ ]:
engineered = ['TotalSF', 'TotalBath', 'TotalPorchSF', 'HouseAge', 'RemodAge',
              'HasPool', 'Has2ndFlr', 'HasGarage', 'HasBsmt', 'HasFireplace']
originals  = ['GrLivArea', 'TotalBsmtSF', 'FullBath', 'OpenPorchSF', 'YearBuilt', 'YearRemodAdd']
corr       = X_all[engineered + originals].corrwith(pd.Series(y_log)).rename('corr with log price').to_frame()
corr['type'] = np.where(corr.index.isin(engineered), 'engineered', 'original')
display(corr.sort_values('corr with log price', key=np.abs, ascending=False).round(3))

In [ ]:
class SkewLog1p(BaseEstimator, TransformerMixin):
    def __init__(self, limit=SKEW_LIMIT):
        self.limit = limit

    def fit(self, X, y=None):
        X          = np.asarray(X, dtype=float)
        self.mask_ = (np.abs(np.nan_to_num(skew(X, axis=0))) > self.limit) & (X.min(axis=0) >= 0)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float).copy()
        X[:, self.mask_] = np.log1p(X[:, self.mask_])
        return X

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)


def fences(values, method, alpha=0.05):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    if method == 'zscore':
        z = norm.ppf(1 - alpha / 2)
        return values.mean() - z * values.std(), values.mean() + z * values.std()
    if method == 'iqr':
        q1, q3 = np.percentile(values, [25, 75])
        return q1 - IQR_K * (q3 - q1), q3 + IQR_K * (q3 - q1)
    return tuple(np.quantile(values, WINSOR_LIMITS))


class OutlierTrimmer(BaseEstimator, TransformerMixin):
    def __init__(self, method=None, columns=None, all_columns=None):
        self.method      = method
        self.columns     = columns
        self.all_columns = all_columns

    def fit(self, X, y=None):
        X           = np.asarray(X, dtype=float)
        names       = list(self.all_columns)
        self.rules_ = [] if self.method is None else [
            (names.index(col), *fences(X[:, names.index(col)], self.method, alpha))
            for col, alpha in self.columns.items() if col in names]
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float).copy()
        for i, low, high in self.rules_:
            if self.method == 'winsorize':
                X[:, i] = np.clip(X[:, i], low, high)
            else:
                X[(X[:, i] < low) | (X[:, i] > high), i] = np.nan
        return X

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)


def make_preprocessor(trim=TRIM_OUTLIERS):
    numeric     = make_pipeline(OutlierTrimmer(trim, TRIM_COLUMNS, num_cols), SimpleImputer(strategy='median'),
                                SkewLog1p(), StandardScaler())
    categorical = make_pipeline(SimpleImputer(strategy='constant', fill_value='None'),
                                OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=MIN_CATEGORY_COUNT,
                                              sparse_output=False))
    return ColumnTransformer([('num', numeric, num_cols), ('cat', categorical, cat_cols)], verbose_feature_names_out=False)


preview_prep = make_preprocessor().fit(X_all)
log_columns  = [col for col, flag in zip(num_cols, preview_prep.named_transformers_['num'].named_steps['skewlog1p'].mask_) if flag]
show_block('Outputs', {'model matrix (all train)': preview_prep.transform(X_all).shape,
                       'numeric columns logged':   len(log_columns),
                       'one-hot columns':          len(preview_prep.named_transformers_['cat'][1].get_feature_names_out())})

# [7/12] Multiple linear regression

Score every model with the same 5-fold cross-validation → all 79 features beat one, but plain least squares starts to wobble

**Question:** How much better than one feature is a regression on all of them?

$$\hat y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_p x_p \qquad \hat\beta = (X^\top X)^{-1} X^\top y$$
- $x_1 \dots x_p$ → the $p$ prepared columns of one house, $X$ → all houses stacked as rows
- $\beta_j$ → price change when $x_j$ moves by one unit **and every other column stays fixed**
- on log price, $e^{\beta_j} - 1$ → the **percent** change in price
- $(X^\top X)^{-1}$ → undoes the overlap between columns; when columns repeat each other it divides by nearly zero and the $\beta$s explode
- Analogy → a restaurant bill: total = Σ (price of each dish × how many you ordered). With every dish's price on the menu the bill is easy; regression works backwards from many bills to the menu prices

**Cross-validation** → split the houses into `N_FOLDS` groups, train on 4, score on the 5th, rotate. Every house is scored exactly once by a model that never saw it, like a teacher giving each fifth of the class an exam written from the other four fifths' notes.

**Answer:** Much better, most of the time. All features cut the typical miss from 49% (always guessing the average) and 32% (area only) to about **13%** on 4 of the 5 folds (RMSE 0.12-0.13). Fold 5 collapses to 0.54: plain least squares priced a \$274,970 house at **\$187,858,413** and a \$250,000 house at **\$689**. With 249 overlapping columns, a few rare category combinations get extreme coefficients that cost nothing in training and explode on one unseen house → the average (0.208) looks acceptable, but the fold spread (0.164) says this model can't be trusted.

In [ ]:
folds       = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
results     = {}
oof_preds   = {}
fold_scores = {}


def cv_rmse(model, X=X_all, y=y_log):
    oof       = cross_val_predict(model, X, y, cv=folds)
    fold_rmse = [rmse(y[val], oof[val]) for _, val in folds.split(X)]
    return fold_rmse, oof


def evaluate(name, model):
    fold_rmse, oof    = cv_rmse(model)
    mean              = float(np.mean(fold_rmse))
    results[name]     = {'CV RMSE (log)': mean, 'fold std': float(np.std(fold_rmse)), 'typical miss %': (np.exp(mean) - 1) * 100}
    oof_preds[name]   = oof
    fold_scores[name] = fold_rmse
    return oof

In [ ]:
first_models = {
    'Baseline (average price)': DummyRegressor(),
    'SLR (GrLivArea)':          make_pipeline(ColumnTransformer([('x', 'passthrough', ['GrLivArea'])]), LinearRegression()),
    'MLR (all features)':       make_pipeline(make_preprocessor(), LinearRegression()),
}
for i, (name, model) in enumerate(first_models.items(), 1):
    print(f'[{i}/{len(first_models)}] {name}')
    evaluate(name, model)
display(pd.DataFrame(results).T.round(4))

In [ ]:
mlr_full  = make_pipeline(make_preprocessor(), LinearRegression()).fit(X_all, y_log)
mlr_coefs = pd.Series(mlr_full[-1].coef_, index=mlr_full[0].get_feature_names_out())
show_block('Outputs', {'coefficients':            len(mlr_coefs),
                       'largest |β|':             f'{mlr_coefs.abs().max():,.2f}',
                       'median |β|':              f'{mlr_coefs.abs().median():,.4f}'})
display(pd.DataFrame(fold_scores, index=[f'fold {i}' for i in range(1, N_FOLDS + 1)]).T.round(4))

In [ ]:
mlr_miss  = np.abs(oof_preds['MLR (all features)'] - y_log)
worst     = train.loc[mlr_miss.argsort()[::-1][:3], ['Id', 'GrLivArea', 'Neighborhood', 'SaleCondition', TARGET]]
worst['MLR predicted $'] = np.expm1(oof_preds['MLR (all features)'][worst.index]).round(0)
display(worst)

# [8/12] Ridge and Lasso

Add a penalty on large coefficients and tune its strength by cross-validation → steadier coefficients, a lower error, and Lasso keeps only part of the columns

**Question:** Can a penalty on large coefficients calm the overlapping columns down?

$$\text{Ridge:}\quad \min_\beta \sum_{i}(y_i - \hat y_i)^2 + \alpha \sum_{j}\beta_j^2 \qquad\qquad \text{Lasso:}\quad \min_\beta \frac{1}{2n}\sum_{i}(y_i - \hat y_i)^2 + \alpha \sum_{j}|\beta_j|$$
- first term → the usual squared misses
- $\alpha$ → **penalty strength**: 0 = plain least squares, large = every $\beta$ pushed towards 0
- $\beta_j^2$ (Ridge, **L2**) → big coefficients cost a lot, small ones almost nothing → all coefficients shrink, none disappear. Overlapping columns share the credit
- $|\beta_j|$ (Lasso, **L1**) → every coefficient costs the same per unit → weak columns are pushed to **exactly 0** = built-in feature selection
- Analogy → Ridge is a tax that grows with income: everyone keeps something, the rich keep relatively less. Lasso is an airline baggage limit: you leave whole items at home rather than trimming each one
- columns are standardised first, so the penalty treats square feet and bathroom counts fairly

**Answer:** Yes. Both curves have a clear valley: Ridge is best at α ≈ 14.7, Lasso at α ≈ 0.00056. Cross-validated RMSE drops from 0.208 (plain MLR) to **0.113** (Ridge) and **0.112** (Lasso), a typical miss of about 12%, and the fold spread falls from 0.164 to below 0.007: no more exploding houses. Too little penalty lets the noise back in, too much flattens real effects; the valley between them is the bias-variance trade-off made visible.

In [ ]:
def alpha_curve(estimator, grid):
    scores = []
    for i, alpha in enumerate(grid, 1):
        fold_rmse, _ = cv_rmse(make_pipeline(make_preprocessor(), clone(estimator).set_params(alpha=alpha)))
        scores.append(np.mean(fold_rmse))
    return np.array(scores)


ridge_grid   = np.logspace(*RIDGE_ALPHAS)
lasso_grid   = np.logspace(*LASSO_ALPHAS)
print(f'[1/2] Ridge · {len(ridge_grid)} alphas × {N_FOLDS} folds')
ridge_scores = alpha_curve(Ridge(), ridge_grid)
print(f'[2/2] Lasso · {len(lasso_grid)} alphas × {N_FOLDS} folds')
lasso_scores = alpha_curve(Lasso(max_iter=50000), lasso_grid)
best_ridge   = ridge_grid[np.argmin(ridge_scores)]
best_lasso   = lasso_grid[np.argmin(lasso_scores)]
show_block('Outputs', {'best Ridge alpha': f'{best_ridge:.4g}', 'best Lasso alpha': f'{best_lasso:.4g}'})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, name, grid, scores, best in [(axes[0], 'Ridge', ridge_grid, ridge_scores, best_ridge),
                                      (axes[1], 'Lasso', lasso_grid, lasso_scores, best_lasso)]:
    ax.semilogx(grid, scores, marker='o', markersize=4, color=PALETTE['main'])
    ax.axvline(best, color=PALETTE['flag'], linestyle='--', label=f'best α = {best:.3g}')
    ax.axhline(results['MLR (all features)']['CV RMSE (log)'], color=PALETTE['soft'], linestyle=':', label='plain MLR')
    ax.set(xlabel='α (log scale)', ylabel='CV RMSE (log price)', title=f'{name}: error vs penalty strength')
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
RIDGE_NAME = f'Ridge (α = {best_ridge:.3g})'
LASSO_NAME = f'Lasso (α = {best_lasso:.3g})'
evaluate(RIDGE_NAME, make_pipeline(make_preprocessor(), Ridge(alpha=best_ridge)))
evaluate(LASSO_NAME, make_pipeline(make_preprocessor(), Lasso(alpha=best_lasso, max_iter=50000)))
display(pd.DataFrame(results).T.round(4))

**Question:** Which features does Lasso keep, and what is each worth?

Columns are standardised, so a numeric coefficient reads "**one standard deviation more** of this feature changes the price by $e^{\beta} - 1$". A one-hot coefficient reads "being in this category vs the reference".

**Answer:** Lasso keeps **100 of 249** columns and sets 149 to exactly zero. The strongest effects make real-estate sense:
- **Zoning** → commercial zoning (`MSZoning = C (all)`) cuts the price by about 24%
- **Location** → Crawford +10%, Stone Brook +9%, Northridge Heights +5% relative to the other neighbourhoods
- **Size and quality** → one standard deviation more living area +9%, overall quality +7%, total area +5%
- **Sale** → new houses +7%, abnormal sales (foreclosures, short sales) −3.5%
- **Age** → one standard deviation older −4%

Each effect is "all else equal": `GrLivArea` and `TotalSF` share the credit for size, so each looks smaller than size as a whole.

In [ ]:
lasso_full  = make_pipeline(make_preprocessor(), Lasso(alpha=best_lasso, max_iter=50000)).fit(X_all, y_log)
lasso_coefs = pd.Series(lasso_full[-1].coef_, index=lasso_full[0].get_feature_names_out())
top_coefs   = lasso_coefs[lasso_coefs != 0].sort_values(key=np.abs, ascending=False).head(20)
show_block('Outputs', {'columns in': len(lasso_coefs), 'kept (β ≠ 0)': int((lasso_coefs != 0).sum()),
                       'dropped (β = 0)': int((lasso_coefs == 0).sum())})
display(pd.DataFrame({'β (log price)': top_coefs, 'price effect %': (np.exp(top_coefs) - 1) * 100}).round(3))

In [ ]:
plot_coefs = top_coefs.sort_values()
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(plot_coefs.index, (np.exp(plot_coefs) - 1) * 100,
        color=[PALETTE['flag'] if value < 0 else PALETTE['main'] for value in plot_coefs])
ax.set(xlabel='price effect (%)', title='Lasso: 20 strongest features')
plt.show()

# [9/12] Benchmark

Give a gradient-boosted tree model the same features and folds → compare it with Lasso, then average the two

**Question:** Does a tree model that needs no linearity beat the tuned linear model?

$$F_m(x) = F_{m-1}(x) + \nu \, h_m(x)$$
- $F_{m-1}$ → the prediction after $m-1$ trees
- $h_m$ → a small new tree fitted to what $F_{m-1}$ still gets wrong (the residuals)
- $\nu$ → **learning rate**: each tree only fixes a small share of the remaining error
- Analogy → a house valuation passed along a row of appraisers: each one only adjusts the previous estimate for something the others missed (a pool, a busy road), and nobody overrides the whole number

**Blend** → the average of Lasso and LightGBM log predictions. They make different kinds of mistakes (straight lines vs step functions), so averaging cancels part of both, like averaging two independent estimates of a house's value.

**Answer:** Not on its own: LightGBM scores **0.116**, slightly behind Lasso's 0.112. After the log and the formula-driven transforms, house prices are close to linear, and 1,458 houses are few for a tree model to learn every step from. The **blend** is the best model: **0.109**, a typical miss of 11.6%, because the two models' errors only partly overlap.

In [ ]:
print('[1/2] LightGBM')
evaluate('LightGBM', make_pipeline(make_preprocessor(), LGBMRegressor(random_state=SEED, **LGBM_PARAMS)))
print('[2/2] Blend (Lasso + LightGBM) / 2')
blend_oof  = (oof_preds[LASSO_NAME] + oof_preds['LightGBM']) / 2
blend_rmse = [rmse(y_log[val], blend_oof[val]) for _, val in folds.split(X_all)]
results['Blend (Lasso + LightGBM)']   = {'CV RMSE (log)': np.mean(blend_rmse), 'fold std': np.std(blend_rmse),
                                         'typical miss %': (np.exp(np.mean(blend_rmse)) - 1) * 100}
oof_preds['Blend (Lasso + LightGBM)'] = blend_oof
leaderboard = pd.DataFrame(results).T.sort_values('CV RMSE (log)')
display(leaderboard.round(4))

In [ ]:
ranked = leaderboard.drop(index='Baseline (average price)').sort_values('CV RMSE (log)', ascending=False)
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.barh(ranked.index, ranked['CV RMSE (log)'], xerr=ranked['fold std'], capsize=3,
        color=[PALETTE['flag'] if name == ranked.index[-1] else PALETTE['soft'] for name in ranked.index])
ax.set(xlabel=f'CV RMSE on log price (baseline = {results["Baseline (average price)"]["CV RMSE (log)"]:.3f})',
       title='Model comparison, same 5 folds (lower is better)')
plt.show()

# [10/12] Outlier treatment

Re-test the earlier version's z-score trimmer next to 5 alternatives, same Lasso, same folds → every trimmer makes the model worse; the log transform and the Lasso penalty already tame the extremes

**Question:** The earlier version of this notebook trimmed 5 features (plus the price) with a z-score. What exactly does each treatment flag, feature by feature?

**1. Z-score → median** (the earlier version)

$$z_i = \frac{x_i - \mu}{\sigma} \qquad \text{flag when } |z_i| > z_{\alpha/2} \quad (1.96 \text{ for } \alpha = 0.05)$$
- $x_i$ → the value of house $i$, $\mu$ → the column's average, $\sigma$ → its standard deviation
- $z_i$ → how many standard deviations house $i$ sits from the average
- $\alpha$ → the share of houses a **bell curve** would push past the fence; flagged values become missing and are filled with the median
- Analogy → the height of school children: a child 2 standard deviations above the class average is rare and worth a second look. Applied to incomes in a town, the same rule calls every doctor "an error"

**2. IQR → median** (Tukey's fences)

$$\text{flag when } x_i < Q_1 - k \cdot \text{IQR} \;\text{ or }\; x_i > Q_3 + k \cdot \text{IQR} \qquad \text{IQR} = Q_3 - Q_1, \; k = 1.5$$
- $Q_1, Q_3$ → the values a quarter and three quarters of the way up the sorted column, IQR → the width of the middle half
- built from quartiles, so the extremes can't drag the fences outwards the way they drag $\mu$ and $\sigma$
- Analogy → a queue at a bus stop: the middle half of the queue sets its length, and anyone standing more than 1.5 queue-lengths away is not in the queue

**3. Winsorize** (capping)

$$x_i \;\to\; \min\big(\max(x_i, P_{1}), P_{99}\big)$$
- $P_1, P_{99}$ → the 1st and 99th percentiles: values beyond them are **pulled back** to the edge instead of removed
- Analogy → a bathroom scale that tops out at 150 kg: heavier people read 150, they don't vanish from the record

**4-5. Drop rows** → remove the whole house from the **training** folds when any of its 5 features is flagged by the z-score; variant 5 also drops by `SalePrice` z-score, as the earlier version did. Validation houses are never removed: the test set will contain mansions too.

**6. Huber loss** → no trimming at all; the model itself stops listening to large errors (formula below)

Every fence is computed inside each training fold only, like every other fitted step.

**Answer:** The two fences disagree, because none of these columns is a bell curve:
- `LotArea` (skewness 12.6) → the z-score's lower fence is **−8,857 sq ft**, a lot size that can't exist, so it only ever flags big lots (21). The IQR, built from the middle half, flags 66
- `TotalBsmtSF` → the z-score flags 86 houses, the most of any column, and 37 of them are simply houses **without a basement**: 0 sq ft sits below its lower fence of 240 sq ft. A missing basement is a fact, not a typo
- `GrLivArea` / `1stFlrSF` → here it flips: the z-score flags 70 / 59 houses, the IQR only 29 / 18
- `LotFrontage` → α = 0.01 still flags 19 lots above 130 ft, on top of the 259 already missing
- **Winsorize** → always touches about 1-2% per column (12-30 values) and keeps them, just pulled to the edge

The histograms show the cost of "→ median": the replaced values stack into a **spike at the median**. The 4,476 sq ft house that sold for \$745,000 enters the model as an ordinary ~1,430 sq ft house.

In [ ]:
treatments = ['zscore', 'iqr', 'winsorize']
flag_rows  = {}
for col, alpha in TRIM_COLUMNS.items():
    values = train[col].to_numpy(dtype=float)
    row    = {'skewness': skew(values[~np.isnan(values)])}
    for method in treatments:
        low, high          = fences(values, method, alpha)
        row[f'{method} low']  = low
        row[f'{method} high'] = high
        row[f'{method} n']    = int(((values < low) | (values > high)).sum())
    flag_rows[col] = row
flags = pd.DataFrame(flag_rows).T
display(flags.round(1))

In [ ]:
fig, axes = plt.subplots(len(TRIM_COLUMNS), 4, figsize=(18, 3.2 * len(TRIM_COLUMNS)))
for r, (col, alpha) in enumerate(TRIM_COLUMNS.items()):
    values = train[col].to_numpy(dtype=float)
    values = values[~np.isnan(values)]
    z_low, z_high = fences(values, 'zscore', alpha)
    i_low, i_high = fences(values, 'iqr')
    w_low, w_high = fences(values, 'winsorize')
    bins          = np.linspace(values.min(), values.max(), 50)

    axes[r, 0].hist(values, bins=bins, color=PALETTE['soft'])
    for x in (z_low, z_high):
        axes[r, 0].axvline(x, color=PALETTE['flag'], linestyle='--')
    for x in (i_low, i_high):
        axes[r, 0].axvline(x, color=PALETTE['main'], linestyle=':')
    axes[r, 0].set(title=f'{col} · before (red z-score, black IQR fences)', xlim=(values.min(), values.max()))

    for c, (name, low, high) in enumerate([('z-score → median', z_low, z_high), ('IQR → median', i_low, i_high)], 1):
        out     = (values < low) | (values > high)
        trimmed = np.where(out, np.median(values[~out]), values)
        axes[r, c].hist(trimmed, bins=bins, color=PALETTE['main'])
        axes[r, c].set(title=f'after {name} · {out.sum()} values replaced', xlim=(values.min(), values.max()))

    capped = np.clip(values, w_low, w_high)
    axes[r, 3].hist(capped, bins=bins, color=PALETTE['main'])
    axes[r, 3].set(title=f'after winsorize · {int(((values < w_low) | (values > w_high)).sum())} values capped',
                   xlim=(values.min(), values.max()))
plt.tight_layout()
plt.show()

**Question:** Are the flagged houses broken records or real houses?

**Answer:** Real houses, and mostly the expensive ones. On every scatter the red points continue the same upward band as the grey ones instead of falling off it. Together the 5 z-score rules flag **178 houses (12%)**: their median price is **\$251,646** against \$158,000 for the rest, and they include every house above \$451,950. The earlier version wasn't removing errors; it was removing the top of the market, and the test set has a top of the market too.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, ['GrLivArea', 'TotalBsmtSF', 'LotArea']):
    values    = train[col].to_numpy(dtype=float)
    low, high = fences(values, 'zscore', TRIM_COLUMNS[col])
    out       = (values < low) | (values > high)
    ax.scatter(values[~out], y_log[~out], s=8, alpha=0.4, color=PALETTE['soft'], label='kept')
    ax.scatter(values[out], y_log[out], s=20, color=PALETTE['flag'], label=f'flagged by z-score ({out.sum()})')
    ax.set(xlabel=col, ylabel='log price', title=f'{col}: flagged houses vs the rest')
    ax.legend()
plt.tight_layout()
plt.show()

z_flagged = np.zeros(len(train), dtype=bool)
for col, alpha in TRIM_COLUMNS.items():
    values     = train[col].to_numpy(dtype=float)
    low, high  = fences(values, 'zscore', alpha)
    z_flagged |= (values < low) | (values > high)
display(pd.DataFrame({'flagged': train.loc[z_flagged, TARGET].describe(),
                      'kept':    train.loc[~z_flagged, TARGET].describe()}).T.round(0))

**Question:** Treatment 6 doesn't touch the data. How can a loss function ignore outliers?

$$L_\delta(r) = \begin{cases} \tfrac12 r^2 & |r| \le \delta \\ \delta\,\big(|r| - \tfrac12\delta\big) & |r| > \delta \end{cases}$$
- $r$ → the residual of one house (real minus predicted, log price)
- $\delta$ → the switch point (`epsilon` = 1.35 robust standard deviations in scikit-learn)
- small misses → squared, exactly like least squares; large misses → grow only **linearly**, so one strange sale can't pull the whole line towards itself
- the penalty strength $\alpha$ (an L2 penalty, as in Ridge) is tuned over `HUBER_ALPHAS` with the same folds
- Analogy → a jury that listens to every witness, but a witness who shouts gets no extra say past a certain volume

**Answer:** Huber is close to Lasso: its best α is about **56**, and its error **0.1117** sits next to Lasso's 0.1120. With the log price and the 2 partial-sale giants already gone, very few residuals are large enough to reach the linear part of the loss, so a robust loss has little left to protect against.

In [ ]:
r = np.linspace(-4, 4, 200)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(r, r ** 2 / 2, color=PALETTE['soft'], label='squared (least squares, Ridge, Lasso)')
ax.plot(r, np.where(np.abs(r) <= 1.35, r ** 2 / 2, 1.35 * (np.abs(r) - 0.675)), color=PALETTE['flag'], label='Huber, δ = 1.35')
ax.set(xlabel='residual r', ylabel='loss', title='How much one miss costs', ylim=(0, 8))
ax.legend()
plt.show()

In [ ]:
huber_grid   = np.logspace(*HUBER_ALPHAS)
huber_scores = alpha_curve(HuberRegressor(max_iter=2000), huber_grid)
best_huber   = huber_grid[np.argmin(huber_scores)]
display(pd.DataFrame({'alpha': huber_grid, 'CV RMSE (log)': huber_scores}).round(4).T)

**Question:** Which treatment actually lowers the cross-validated error?

All 7 rows use Lasso at the α tuned in [8/12] (Huber uses its own α) and the same 5 folds, so the only thing that changes is how outliers are treated.

**Answer:** None of them, and the more a treatment deletes, the worse it gets:
- **Z-score → median (earlier version)** → 0.1141, **+0.0021** worse than leaving the values alone (0.1120); IQR → median is similar (+0.0024)
- **Winsorize** → +0.0008, the gentlest and the least harmful: it keeps the order of the houses, only shortens the tail
- **Drop rows** → the worst: about 142 of ~1,166 training houses lost per fold, error **0.1207 (+0.0087)** and a fold spread almost twice as wide. Also dropping by `SalePrice` (157 houses) scores 0.1193: the model never sees an expensive house, then has to price them in validation
- **Huber loss** → 0.1117, the only row below "none" (−0.0003), but that gap is far smaller than the fold-to-fold spread (std 0.0085, wider than Lasso's 0.0064) → a tie, not a win

Why: `log1p` already squeezes the long tails (31 skewed columns, [6/12]), the price is modelled in log, and the Lasso penalty stops any single house from inflating a coefficient. Each trimmer only removes **true information** about big houses on top of that. → `TRIM_OUTLIERS` stays `None`, and the targeted rule from [4/12] (2 partial-sale giants) remains the only removal. Set `TRIM_OUTLIERS` in the top cell to rerun every model with a trimmer.

In [ ]:
def cv_drop_rows(model, with_target=False):
    oof, dropped = np.zeros(len(y_log)), []
    for tr, val in folds.split(X_all):
        keep = np.ones(len(tr), dtype=bool)
        for col, alpha in TRIM_COLUMNS.items():
            values     = X_all[col].to_numpy(dtype=float)[tr]
            low, high  = fences(values, 'zscore', alpha)
            keep      &= ~((values < low) | (values > high))
        if with_target:
            prices     = train[TARGET].to_numpy(dtype=float)[tr]
            low, high  = fences(prices, 'zscore', 0.05)
            keep      &= (prices >= low) & (prices <= high)
        oof[val] = clone(model).fit(X_all.iloc[tr[keep]], y_log[tr[keep]]).predict(X_all.iloc[val])
        dropped.append((~keep).sum())
    return [rmse(y_log[val], oof[val]) for _, val in folds.split(X_all)], oof, float(np.mean(dropped))


lasso_for = lambda trim: make_pipeline(make_preprocessor(trim), Lasso(alpha=best_lasso, max_iter=50000))
trials    = {
    'None (main notebook)':            lambda: (*cv_rmse(lasso_for(None)), 0),
    'Z-score → median (earlier)':      lambda: (*cv_rmse(lasso_for('zscore')), 0),
    'IQR → median':                    lambda: (*cv_rmse(lasso_for('iqr')), 0),
    'Winsorize 1-99%':                 lambda: (*cv_rmse(lasso_for('winsorize')), 0),
    'Drop rows: z-score features':     lambda: cv_drop_rows(lasso_for(None)),
    'Drop rows: features + SalePrice': lambda: cv_drop_rows(lasso_for(None), with_target=True),
    f'Huber loss (α = {best_huber:.3g})': lambda: (*cv_rmse(make_pipeline(make_preprocessor(None),
                                                                     HuberRegressor(alpha=best_huber, max_iter=2000))), 0),
}
trim_results = {}
for i, (name, run) in enumerate(trials.items(), 1):
    print(f'[{i}/{len(trials)}] {name}')
    fold_rmse, _, dropped = run()
    trim_results[name] = {'CV RMSE (log)': np.mean(fold_rmse), 'fold std': np.std(fold_rmse),
                          'train rows dropped per fold': dropped}
trim_table = pd.DataFrame(trim_results).T
trim_table['vs none'] = trim_table['CV RMSE (log)'] - trim_table.loc['None (main notebook)', 'CV RMSE (log)']
display(trim_table.round(4))

In [ ]:
ranked_trim = trim_table.sort_values('CV RMSE (log)', ascending=False)
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.barh(ranked_trim.index, ranked_trim['CV RMSE (log)'],
        color=[PALETTE['flag'] if name == 'None (main notebook)' else PALETTE['soft'] for name in ranked_trim.index])
ax.axvline(trim_table.loc['None (main notebook)', 'CV RMSE (log)'], color=PALETTE['flag'], linestyle='--')
for y, (score, change) in enumerate(zip(ranked_trim['CV RMSE (log)'], ranked_trim['vs none'])):
    ax.text(score + 0.0003, y, f'{score:.4f} ({change:+.4f})', va='center', fontsize=9)
ax.set(xlabel='CV RMSE on log price (lower is better)', title='Outlier treatments, same Lasso, same 5 folds',
       xlim=(ranked_trim['CV RMSE (log)'].min() * 0.97, ranked_trim['CV RMSE (log)'].max() * 1.05))
plt.show()

# [11/12] Diagnostics

Inspect the out-of-fold errors of the blend → which houses are still hard to price, and why

**Question:** Where does the final model still miss, and is there a pattern?

**Answer:** For most houses the blend tracks reality closely: the median miss is **5.0%** on normal sales. Every one of the 8 largest misses is an **unusual sale** (abnormal, family or partial) or a house under \$65,000 in IDOTRR, Sawyer, OldTown or Edwards, and the model priced all of them 70-122% too high. The median miss is 10.0% for abnormal sales and 12.1% for sales between family members: those prices reflect the circumstances of the sale, which no column fully describes.

In [ ]:
best_oof = oof_preds['Blend (Lasso + LightGBM)']
diag = pd.DataFrame({'Id':            train['Id'],
                     'GrLivArea':     train['GrLivArea'],
                     'OverallQual':   train['OverallQual'],
                     'Neighborhood':  train['Neighborhood'],
                     'SaleCondition': train['SaleCondition'],
                     'actual $':      train[TARGET],
                     'predicted $':   np.expm1(best_oof).round(0)})
diag['miss %'] = ((diag['predicted $'] / diag['actual $'] - 1) * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
axes[0].scatter(diag['actual $'], diag['predicted $'], s=8, alpha=0.5, color=PALETTE['main'])
axes[0].plot([diag['actual $'].min(), diag['actual $'].max()], [diag['actual $'].min(), diag['actual $'].max()], color=PALETTE['flag'])
axes[0].set(xlabel='actual price ($)', ylabel='predicted price ($)', title='Out-of-fold predictions vs reality')
axes[1].scatter(best_oof, y_log - best_oof, s=8, alpha=0.5, color=PALETTE['main'])
axes[1].axhline(0, color=PALETTE['flag'])
axes[1].set(xlabel='predicted log price', ylabel='residual (log)', title='Residuals vs predicted')
plt.tight_layout()
plt.show()

In [ ]:
display(diag.reindex(diag['miss %'].abs().sort_values(ascending=False).index).head(8))
display(diag.assign(abs_miss=diag['miss %'].abs()).groupby('SaleCondition')['abs_miss']
        .agg(['median', 'size']).rename(columns={'median': 'median |miss %|', 'size': 'houses'}).round(1))

# [12/12] Submission

Refit Lasso and LightGBM on every training house and price the test houses → preview by default, CSV only on request

**Question:** How would this score on the leaderboard?

**Answer:** Set `SAVE_SUBMISSION = True` to write `submission.csv` for all 1,459 test houses. The cross-validated blend error above is the best local estimate of the leaderboard score; predictions are converted back from log with $e^{\tilde y} - 1$.

In [ ]:
"""Optional · Set SAVE_SUBMISSION = True in the top cell to write the CSV."""
show_block('Inputs', {'mode':        'save' if SAVE_SUBMISSION else 'preview',
                      'train rows':  len(X_all),
                      'test rows':   len(X_test),
                      'output file': SUBMISSION_FILE})

In [ ]:
final_lasso = make_pipeline(make_preprocessor(), Lasso(alpha=best_lasso, max_iter=50000)).fit(X_all, y_log)
final_lgbm  = make_pipeline(make_preprocessor(), LGBMRegressor(random_state=SEED, **LGBM_PARAMS)).fit(X_all, y_log)
test_log    = (final_lasso.predict(X_test) + final_lgbm.predict(X_test)) / 2
submission  = pd.DataFrame({'Id': test_raw['Id'], TARGET: np.expm1(test_log)})
display(submission.head(SUBMISSION_PREVIEW))
show_block('Outputs', {'predicted median price': f'${submission[TARGET].median():,.0f}',
                       'predicted range':        f'${submission[TARGET].min():,.0f} - ${submission[TARGET].max():,.0f}'})

In [ ]:
if SAVE_SUBMISSION:
    submission.to_csv(SUBMISSION_FILE, index=False)
    print(f'Saved → {SUBMISSION_FILE}')
else:
    print(f'Preview · {len(submission)} rows · nothing written (SAVE_SUBMISSION = False)')

**Next steps**

- **Leaderboard check** → submit with `SAVE_SUBMISSION = True` and compare the public RMSE with the cross-validated 0.109
- **Unusual sales** → flag or model abnormal, family and partial sales separately: they hold the biggest misses
- **Neighbourhood × size** → interaction terms, since a square foot is worth more in NridgHt than in MeadowV
- **Stacking** → learn the blend weights instead of 50 / 50, and add ElasticNet, SVR or CatBoost
- **Inference** → refit the Lasso-selected features with `statsmodels` OLS for a confidence interval on every coefficient